In [ ]:
#| default_exp skill

## Installing the skill

Install the notebook workflow instructions and their reference notes for agents.

The repository also ships the instructions that teach an agent how to use these tools. This notebook installs the bundled `SKILL.md` and references into a local Codex or Claude skills directory so the notebook-first workflow can travel with the package.

The skill files are packaging for agent behavior, not another implementation path. They point agents back to the same notebook-aware tools in this repository, so users get consistent reads, writes, execution, and review whether they invoke nbskill through Python, the CLI, or MCP.

```python
build_skill_from_readme("README.md", "nbskill/SKILL.md")
install_nbskill(dest="~/.codex/skills/jupyter-notebooks")
```

In [ ]:
from contextlib import redirect_stdout as _redirect_stdout
from io import StringIO as _StringIO
from nbskill.skill import build_skill_from_readme as _example_build_skill_from_readme
from nbskill.foundation import demo_path, remove_demo_path, write_demo_notebook

In [ ]:
root = demo_path("06_skill_example")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text("before\n<!-- nbskill-skill:start -->\n# Skill Body\nUse read_nb first.\n<!-- nbskill-skill:end -->\nafter\n", encoding="utf-8")
    _example_build_skill_from_readme(str(readme), str(out))
    print("built:", out.name)
    print(out.read_text(encoding="utf-8").splitlines()[-2])
finally:
    remove_demo_path(root)

Built nbs/data/06_skill_example/SKILL.md from nbs/data/06_skill_example/README.md
built: SKILL.md
# Skill Body


In [ ]:
#| export
import subprocess
from importlib.resources import files
from pathlib import Path

from fastcore.script import call_parse

from nbskill.foundation import cli_return, install_nbdev_pre_commit_hooks, tracked_call

In [ ]:
#| export
_SKILL_START = "<!-- nbskill-skill:start -->"
_SKILL_END = "<!-- nbskill-skill:end -->"
_SKILL_FRONTMATTER = """---
name: jupyter-notebooks
description: Work notebook-first in nbdev projects with nbskill MCP tools for reading, writing, updating, and executing notebooks without raw JSON.
---"""


def _marked_readme_section(text, start=_SKILL_START, end=_SKILL_END):
    try:
        body = text.split(start, 1)[1].split(end, 1)[0]
    except IndexError as exc:
        raise ValueError(f"README is missing {start!r} and {end!r} markers") from exc
    return body.strip()


def _skill_frontmatter(out_path):
    path = Path(out_path)
    if path.exists():
        text = path.read_text(encoding="utf-8")
        if text.startswith("---\n"):
            parts = text.split("\n---", 1)
            if len(parts) == 2: return f"---{parts[0][3:]}\n---"
    return _SKILL_FRONTMATTER


@call_parse
@tracked_call
def build_skill_from_readme(
    readme_path: str = "README.md",  # README containing the marked skill section
    out_path: str = "nbskill/SKILL.md",  # Skill file to write
):
    "Build SKILL.md from the marked section of README.md."
    out = Path(out_path)
    body = _marked_readme_section(Path(readme_path).read_text(encoding="utf-8"))
    text = f"{_skill_frontmatter(out)}\n\n{body}\n"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(text, encoding="utf-8")
    print(f"Built {out} from {readme_path}")
    return cli_return(out)

### Installing agent instructions

The installer resolves the target skills directory, copies the packaged `SKILL.md`, and includes reference files. That keeps the project documentation and the agent workflow in sync with the package version.

In [ ]:
#| export
def _nbskill_mcp_pids():
    proc = subprocess.run(["ps", "-eo", "pid=,command="], text=True, capture_output=True)
    if proc.returncode != 0: return []
    pids = []
    for line in proc.stdout.splitlines():
        line = line.strip()
        if not line: continue
        pid_text, _, command = line.partition(" ")
        if "nbskill_mcp" not in command and "nbskill.mcp" not in command: continue
        try: pid = int(pid_text)
        except ValueError: continue
        pids.append({"pid": pid, "command": command.strip()})
    return pids


def nbskill_mcp_restart_notice():
    "Return restart guidance for running nbskill MCP server processes without stopping them."
    running = _nbskill_mcp_pids()
    if not running: return {"running": False, "processes": [], "message": "No running nbskill MCP server process detected."}
    message = (
        "A running nbskill MCP server is still using the old code. "
        "Restart or reconnect the MCP client so it launches the updated nbskill_mcp command. "
        "For development outside Codex, FastMCP can auto-reload when launched with `fastmcp run ... --reload`, "
        "but nbskill will not terminate existing MCP processes for you."
    )
    return {"running": True, "processes": running, "message": message}


@call_parse
@tracked_call
def install_nbskill(
    target: str = "codex",  # codex, claude, both, or custom when skills_dir is set
    skills_dir: str | None = None,  # Parent skills directory; skill is installed below jupyter-notebooks
    skill_name: str = "jupyter-notebooks",  # Skill folder name
    overwrite: bool = True,  # Overwrite an existing SKILL.md
    install_hooks: bool = False,  # Install nbdev-clean/nbdev-test pre-commit hooks in the current repo
    restart_mcp: bool = True,  # Detect running Codex nbskill MCP server and print restart guidance
):
    "Install the bundled SKILL.md into a Codex or Claude Code skills directory."
    target = target.lower()
    if skills_dir: roots = [Path(skills_dir).expanduser()]
    elif target == "codex": roots = [Path.home() / ".codex" / "skills"]
    elif target in {"claude", "claude-code", "claude_code"}: roots = [Path.home() / ".claude" / "skills"]
    elif target == "both": roots = [Path.home() / ".codex" / "skills", Path.home() / ".claude" / "skills"]
    else: raise ValueError("target must be codex, claude, both, or use skills_dir")
    package = files("nbskill")
    skill_text = package.joinpath("SKILL.md").read_text(encoding="utf-8")
    references = package.joinpath("references")
    installed = []
    for root in roots:
        dst_dir = root / skill_name
        dst = dst_dir / "SKILL.md"
        if dst.exists() and not overwrite: raise FileExistsError(dst)
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst.write_text(skill_text, encoding="utf-8")
        if references.is_dir():
            ref_dir = dst_dir / "references"
            ref_dir.mkdir(exist_ok=True)
            for ref in references.iterdir():
                if ref.is_file():
                    (ref_dir / ref.name).write_text(ref.read_text(encoding="utf-8"), encoding="utf-8")
        installed.append(dst)
    hooks = install_nbdev_pre_commit_hooks(Path.cwd()) if install_hooks else {"installed": False, "reason": "disabled"}
    mcp_notice = {"running": False, "message": "not checked"}
    if restart_mcp and not skills_dir and target in {"codex", "both"}:
        mcp_notice = nbskill_mcp_restart_notice()
    for path in installed: print(f"Installed {path}")
    if hooks.get("installed"): print(f"Installed nbdev pre-commit hook at {hooks['hook']}")
    else: print(f"Skipped nbdev pre-commit hook: {hooks.get('reason')}")
    if mcp_notice.get("running"):
        print("nbskill MCP restart required:")
        print(mcp_notice["message"])
        for proc in mcp_notice.get("processes", []):
            print(f"- pid={proc['pid']} command={proc['command']}")
    return cli_return({"installed": installed, "hooks": hooks, "mcp_restart": mcp_notice})

In [ ]:
install_root = demo_path("06_skill_install")
try:
    install_nbskill(skills_dir=str(install_root))
    skill_dir = install_root / "jupyter-notebooks"
    assert (skill_dir / "SKILL.md").exists()
    assert (skill_dir / "references" / "mcp-tools.md").exists()
    assert (skill_dir / "references" / "cli-fallbacks.md").exists()
    assert (skill_dir / "references" / "conversion.md").exists()
    assert (skill_dir / "references" / "extended-tools.md").exists()
finally:
    remove_demo_path(install_root)

root = demo_path("06_skill_build")
try:
    root.mkdir()
    readme = root / "README.md"
    out = root / "SKILL.md"
    readme.write_text(
        """before
<!-- nbskill-skill:start -->
# Skill Body
Use notebooks.
<!-- nbskill-skill:end -->
after
""",
        encoding="utf-8",
    )
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: jupyter-notebooks")
    assert "# Skill Body" in text
    assert "before" not in text and "after" not in text
    out.write_text("---\nname: custom-skill\ndescription: Keep me.\n---\nold\n", encoding="utf-8")
    build_skill_from_readme(str(readme), str(out))
    text = out.read_text(encoding="utf-8")
    assert text.startswith("---\nname: custom-skill")
    assert "# Skill Body" in text and "old" not in text
finally:
    remove_demo_path(root)

Installed nbs/data/06_skill_install/jupyter-notebooks/SKILL.md
Built nbs/data/06_skill_build/SKILL.md from nbs/data/06_skill_build/README.md
Built nbs/data/06_skill_build/SKILL.md from nbs/data/06_skill_build/README.md


In [ ]:
import tomllib
from pathlib import Path

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
scripts = tomllib.loads((project_root / "pyproject.toml").read_text(encoding="utf-8"))["project"]["scripts"]
assert "batch_edit_nb" in scripts
assert "nbskill_status" in scripts
assert "install_nbskill" in scripts

for removed in ("nbskill-mcp", "private-symbol-report", "symbol-graph", "update-cell", "show-doc"):
    for path in [project_root / "README.md", project_root / "nbskill/SKILL.md", *(project_root / "nbskill/references").glob("*.md")]:
        if path.exists(): assert removed not in path.read_text(encoding="utf-8")

hook_root = demo_path("06_skill_hooks")
try:
    hook_root.mkdir()
    (hook_root / "nbs").mkdir()
    subprocess.run(["git", "init"], cwd=hook_root, check=True, capture_output=True)
    hook_result = install_nbdev_pre_commit_hooks(hook_root, run_nbdev_install_hooks=False)
    hook_text = (hook_root / ".git" / "hooks" / "pre-commit").read_text(encoding="utf-8")
    assert hook_result["installed"]
    assert "nbdev-clean" in hook_text
    assert "nbdev-test" in hook_text

    install_root = hook_root / "skills"
    install_nbskill(
        target="custom", skills_dir=str(install_root), install_hooks=False, restart_mcp=False,
    )
    assert (install_root / "jupyter-notebooks" / "SKILL.md").exists()
    notice = nbskill_mcp_restart_notice()
    assert "running" in notice and "message" in notice
finally:
    remove_demo_path(hook_root)